## Creation of a Temporary table for all the regions in Beijing via Database Query

## Import Libraries

Load pandas for data manipulation and SQLAlchemy for connecting to the MySQL database.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

## Connect to DATABASE

Create and test a connection to the `air_quality` database.

In [2]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")
except Exception as error:
    print(f"Failed to connect to air_quality: {error}")

Connected to air_quality!


## Combine Station Tables

Load each station table, print its row count, and combine all station data into `df_combined`. The combined data is also exported to `regions.csv`.

In [3]:
table_names = [
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
]

dataframes = []
source_row_counts = {}

for table_name in table_names:
    dataframe = pd.read_sql_query(
        f"SELECT * FROM `{table_name}`",
        engine
    )
    dataframes.append(dataframe)
    source_row_counts[table_name] = len(dataframe)
    print(f"{table_name}: {len(dataframe):,} rows")

df_combined = pd.concat(dataframes, ignore_index=True)

df_combined_raw = df_combined.copy(deep=True)


source_total_rows = sum(source_row_counts.values())
combined_total_rows = len(df_combined)

print(f"\nRows from all source tables: {source_total_rows:,}")
print(f"Rows in df_combined: {combined_total_rows:,}")
print(f"Row counts match: {source_total_rows == combined_total_rows}")

aotizhongxin: 35,064 rows
changping: 35,064 rows
dingling: 35,064 rows
dongsi: 35,064 rows
guanyuan: 35,064 rows
gucheng: 35,064 rows
huairou: 35,064 rows
nongzhanguan: 35,064 rows
shunyi: 35,064 rows
tiantan: 35,064 rows
wanliu: 35,064 rows
wanshouxigong: 35,064 rows

Rows from all source tables: 420,768
Rows in df_combined: 420,768
Row counts match: True


## Check for All Stations

Compare the stations in `df_combined` with the expected list and report missing or unexpected stations.

In [4]:
expected_stations = {
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
}

if "station" not in df_combined.columns:
    raise KeyError("The 'station' column was not found in df_combined.")

present_stations = set(df_combined["station"].dropna().str.lower().unique())
missing_stations = expected_stations - present_stations
unexpected_stations = present_stations - expected_stations

print(f"Expected stations: {len(expected_stations)}")
print(f"Stations found: {len(present_stations)}")
print(f"Missing stations: {sorted(missing_stations) or 'None'}")
print(f"Unexpected stations: {sorted(unexpected_stations) or 'None'}")

if not missing_stations and not unexpected_stations:
    print("All expected stations are present.")
else:
    print("The station list does not exactly match the expected stations.")

Expected stations: 12
Stations found: 12
Missing stations: None
Unexpected stations: None
All expected stations are present.


## Check for Duplicates and Null Values
Review duplicate rows and nulls for each column.

In [5]:
required_id_cols = ["No", "year", "month", "day", "station"]

if not set(required_id_cols).issubset(df_combined.columns):
    missing_required = set(required_id_cols) - set(df_combined.columns)
    raise KeyError(f"Missing required columns: {sorted(missing_required)}")

# Duplicate rows
duplicate_rows = df_combined.duplicated().sum()
print(f"Duplicate rows before cleaning: {duplicate_rows}")

# Required identification columns must not be null
required_nulls = {
    col: int(df_combined[col].isna().sum())
    for col in required_id_cols
}

print("\nNull counts in required identification columns:")
for col, count in required_nulls.items():
    print(f"  {col}: {count}")

# Show null counts for every column with nulls
all_null_counts = df_combined.isna().sum()
null_columns = all_null_counts[all_null_counts > 0]

print("\nNull counts in all columns with null values:")
if null_columns.empty:
    print("  No null values found in any column.")
else:
    for col, count in null_columns.items():
        print(f"  {col}: {count}")

# Hour validation
hour_null_count = int(df_combined["hour"].isna().sum()) if "hour" in df_combined.columns else 0
print(f"\nNull hour count: {hour_null_count}")

if "hour" in df_combined.columns:
    zero_hour_groups = set(
        df_combined.loc[df_combined["hour"] == 0, ["year", "month", "day", "station"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

    null_hour_duplicates = (
        df_combined.loc[df_combined["hour"].isna()]
        .apply(
            lambda row: (
                row["year"], row["month"], row["day"], row["station"]
            ) in zero_hour_groups,
            axis=1
        )
        .sum()
    )

    print(f"Null-hour rows that duplicate an existing 0-hour entry: {null_hour_duplicates}")

# RAIN null count
rain_null_count = int(df_combined["RAIN"].isna().sum()) if "RAIN" in df_combined.columns else 0
print(f"Null RAIN count before fill: {rain_null_count}")

# Excessive missingness check
measurement_cols = [
    col for col in df_combined.columns
    if col not in {"No", "year", "month", "day", "station", "hour", "RAIN"}
]

if measurement_cols:
    missing_ratio = df_combined[measurement_cols].isna().mean(axis=1)
    rows_above_threshold = int((missing_ratio > 0.5).sum())
    print(f"Rows with >50% missing measurement values: {rows_above_threshold}")
else:
    print("No measurement columns available for missingness threshold check.")

Duplicate rows before cleaning: 0

Null counts in required identification columns:
  No: 0
  year: 0
  month: 0
  day: 0
  station: 0

Null counts in all columns with null values:
  PM2.5: 8739
  PM10: 6449
  SO2: 9021
  NO2: 12116
  CO: 20701
  O3: 13277
  TEMP: 398
  PRES: 393
  DEWP: 403
  RAIN: 390
  wd: 1822
  WSPM: 318

Null hour count: 0
Null-hour rows that duplicate an existing 0-hour entry: 0.0
Null RAIN count before fill: 390
Rows with >50% missing measurement values: 4977


## Remove Unacceptable Nulls and Duplicates
Drop invalid rows and fill valid missing values

In [6]:
df_combined = df_combined_raw.copy()

# 1) Remove rows with nulls in required identification columns
required_id_cols = ["No", "year", "month", "day", "station"]
df_combined = df_combined.dropna(subset=required_id_cols).copy()

# 2) Handle hour values
if "hour" in df_combined.columns:
    hour_null_mask = df_combined["hour"].isna()

    if hour_null_mask.any():
        zero_hour_groups = set(
            df_combined.loc[df_combined["hour"] == 0, ["year", "month", "day", "station"]]
            .drop_duplicates()
            .itertuples(index=False, name=None)
        )

        rows_to_drop = (
            df_combined.loc[hour_null_mask]
            .apply(
                lambda row: (
                    row["year"], row["month"], row["day"], row["station"]
                ) in zero_hour_groups,
                axis=1
            )
        )

        df_combined = df_combined.loc[~(hour_null_mask & rows_to_drop)].copy()

        remaining_null_hour_mask = df_combined["hour"].isna()
        if remaining_null_hour_mask.any():
            df_combined.loc[remaining_null_hour_mask, "hour"] = 0

# 3) Replace missing RAIN with 0
if "RAIN" in df_combined.columns:
    df_combined["RAIN"] = df_combined["RAIN"].fillna(0)

# 4) Remove duplicate rows
df_combined = df_combined.drop_duplicates().copy()

# 5) Remove rows with excessive missingness in measurement columns
non_cleaning_cols = {"No", "year", "month", "day", "station", "hour", "RAIN"}
measurement_cols = [col for col in df_combined.columns if col not in non_cleaning_cols]

if measurement_cols:
    missing_ratio = df_combined[measurement_cols].isna().mean(axis=1)
    df_combined = df_combined.loc[missing_ratio <= 0.5].copy()

# Final validation
print(f"Rows after cleaning: {len(df_combined):,}")
print(f"Duplicate rows remaining: {df_combined.duplicated().sum()}")
print(f"Null values in required ID columns remaining: {sum(df_combined[required_id_cols].isna().sum())}")
print(f"Null hour rows remaining: {df_combined['hour'].isna().sum() if 'hour' in df_combined.columns else 0}")
print(f"Null RAIN values remaining: {df_combined['RAIN'].isna().sum() if 'RAIN' in df_combined.columns else 0}")

if measurement_cols:
    remaining_missing_ratio = df_combined[measurement_cols].isna().mean(axis=1)
    print(f"Rows with >50% missing measurements remaining: {(remaining_missing_ratio > 0.5).sum()}")
else:
    print("No measurement columns available for missingness threshold check.")

display(df_combined.head())

Rows after cleaning: 415,791
Duplicate rows remaining: 0
Null values in required ID columns remaining: 0
Null hour rows remaining: 0
Null RAIN values remaining: 0
Rows with >50% missing measurements remaining: 0


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


## Final DataFrame Verification

In [7]:
print(f"Before Cleaning Rows: {len(df_combined_raw):,}")
print(f"Final DataFrame rows: {len(df_combined):,}")
print(f"Final DataFrame columns: {len(df_combined.columns)}")
print(df_combined.head())

Before Cleaning Rows: 420,768
Final DataFrame rows: 415,791
Final DataFrame columns: 18
   No  year  month  day  hour  PM2.5  PM10   SO2   NO2     CO    O3  TEMP  \
0   1  2013      3    1     0    4.0   4.0   4.0   7.0  300.0  77.0  -0.7   
1   2  2013      3    1     1    8.0   8.0   4.0   7.0  300.0  77.0  -1.1   
2   3  2013      3    1     2    7.0   7.0   5.0  10.0  300.0  73.0  -1.1   
3   4  2013      3    1     3    6.0   6.0  11.0  11.0  300.0  72.0  -1.4   
4   5  2013      3    1     4    3.0   3.0  12.0  12.0  300.0  72.0  -2.0   

     PRES  DEWP  RAIN   wd  WSPM       station  
0  1023.0 -18.8   0.0  NNW   4.4  Aotizhongxin  
1  1023.2 -18.2   0.0    N   4.7  Aotizhongxin  
2  1023.5 -18.2   0.0  NNW   5.6  Aotizhongxin  
3  1024.5 -19.4   0.0   NW   3.1  Aotizhongxin  
4  1025.2 -19.5   0.0    N   2.0  Aotizhongxin  


## Load Combined Data into DATABASE

Store `df_combined` in the `air_quality` database 

In [8]:
df_combined.to_sql(
    "all_regions",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

loaded_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM all_regions",
    engine
)

print(f"Rows in final DataFrame: {len(df_combined):,}")
print(f"Rows in MySQL all_regions table: {loaded_count['total_rows'].iloc[0]:,}")
print(
    f"Load row counts match: "
    f"{loaded_count['total_rows'].iloc[0] == len(df_combined)}"
)


Rows in final DataFrame: 415,791
Rows in MySQL all_regions table: 415,791
Load row counts match: True


## Final SQL Analysis

### Question: Which Beijing monitoring station has the highest average PM2.5 concentration?

PM2.5 is fine particulate matter that can be inhaled into the lungs. It is an important indicator of air pollution, so this analysis identifies which monitoring station has the highest average PM2.5 concentration.

In [9]:
query = """
SELECT
    station,
    AVG(`PM2.5`) AS average_pm25
FROM all_regions
GROUP BY station
ORDER BY average_pm25 DESC;
"""

result = pd.read_sql_query(query, engine)

top_5 = result.head(5)
display(top_5)

top_station = result.iloc[0]

print(
    f"\nTop 1 Station: {top_station['station']}"
    f" with an average PM2.5 concentration of "
    f"{top_station['average_pm25']:.2f}"
)

,station,average_pm25
0,Dongsi,86.194297
1,Wanshouxigong,85.024136
2,Nongzhanguan,84.838483
3,Gucheng,83.852089
4,Wanliu,83.374716



Top 1 Station: Dongsi with an average PM2.5 concentration of 86.19
